# 06 · Evaluate the models — one protocol, any prior

This notebook is the evaluation thread's home and the source of every number in the Friday talk.
It compares **every reconstruction method on exactly the same problem**, so that the differences you
report are due to the method and nothing else. It works for the methods in the repo and for any model
you add — a diffusion prior, a power-spectrum prior, a classical solver — through a three-line adapter.

**The protocol** (enforced by `mrigen.evaluate`; state it in your talk):

1. **Held-out slices only.** `FastMRISlices(split="test")` — volumes the prior never saw, and that the
   pre-trained checkpoint never saw either (see `CHECKPOINTS.md`).
2. **Same mask, same noise, same σ for every method**, seeded per (slice, R).
3. **PSNR / SSIM / NMSE** against the fully-sampled ground truth with `data_range = 1`.
4. **Effective acceleration** `R_eff = M.size / M.sum()`, not the nominal R — the ACS band makes 4× really ≈ 3.3×.
5. **Wall-clock seconds** per reconstruction, after one untimed warm-up call (JIT).
6. **Calibration** for every method that returns a per-pixel std: pooled |error| vs predicted std.
7. **Mean ± std over slices.** A difference smaller than the ± is not a result.
8. **Show the worst case**, not only the average.

Runtime: NUTS is the expensive row (up to `2**max_tree_depth` decoder evaluations per sample, and a
poorly conditioned posterior uses all of them). The cell below picks settings from the backend: on the
school's GPU server the full protocol; on a laptop CPU two slices, R ∈ {4, 8}, short chains and a small
tree depth, which takes about ten minutes. Override the constants if you want more.

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp

from mrigen import evaluate as ev
from mrigen import metrics, viz
from mrigen.data import FastMRISlices
from mrigen.recon.spectrum import estimate_power_spectrum, make_spectrum_decoder
from mrigen.train_vae import load_model

# One switch: the GPU server gets the full protocol, a laptop CPU gets a small version of it.
ON_GPU = jax.default_backend() == "gpu"
N_SLICES = 6 if ON_GPU else 2                 # held-out slices to evaluate
ACCELERATIONS = (4, 8, 16) if ON_GPU else (4, 8)
SIGMA = 0.01                                  # measurement noise, shared by every method
NUTS_SAMPLES = 200 if ON_GPU else 40          # per chain (warm-up uses the same number)
NUTS_TREE_DEPTH = 10 if ON_GPU else 5         # NUTS cost is up to 2**depth decoder evaluations per sample
WARMUP = ON_GPU                               # untimed warm-up call per method (JIT); skipped on CPU to save time
print(f"backend: {jax.default_backend()}  ->  {N_SLICES} slices, R = {ACCELERATIONS}, NUTS {NUTS_SAMPLES}/{NUTS_SAMPLES}, tree depth {NUTS_TREE_DEPTH}")

## 1. The data: held-out slices, and training slices for data-driven priors

`split="test"` selects the held-out volumes; `split="train"` everything else. The VAE checkpoint was
trained on the train split; the power-spectrum prior below is estimated from it too. **Never evaluate
on a slice a prior was trained on** — that is the first thing a reviewer (or a judge) will ask.

In [ ]:
test = FastMRISlices('../data/processed', split='test')
train = FastMRISlices('../data/processed', split='train')
print(f"test: {len(test)} slices from {test.volumes}")
print(f"train: {len(train)} slices from {train.volumes}")

# Central slices: the first and last slices of a volume are mostly air and are not representative.
idx = np.linspace(0.3, 0.7, N_SLICES) * (len(test) - 1)
idx = idx.round().astype(int)
images = test.slices[idx]
labels = [f"{test.volumes[test.volume_index[i]]}/{i}" for i in idx]
print("evaluating:", labels)

## 2. The methods — every one an adapter

A **reconstructor** is any callable `recon(y_obs, mask, sigma) -> ev.Recon(mean, std=None, samples=None)`.
`mrigen.evaluate` ships adapters for the repo's methods; the dictionary below is the whole comparison.

| method | prior | uncertainty | adapter |
|---|---|---|---|
| zero-filled | none | — | `ev.zero_filled_recon` |
| TV/L1 (FISTA) | sparsity, hand-made | — | `ev.tv_recon(lam, n_iter)` |
| power spectrum (Wiener) | stationary Gaussian, **learnt** from the train split | closed form (spatially constant) | `ev.wiener_recon(P)` |
| VAE · MAP | β-VAE decoder, learnt | — | `ev.map_recon(vae.decoder, vae.latent_dim)` |
| VAE · posterior (NUTS) | β-VAE decoder, learnt | per-pixel std from samples | `ev.posterior_recon(vae.decoder, vae.latent_dim)` |

The power-spectrum prior is there on purpose: it is the simplest possible *learned* prior — one line,
no training loop — and a useful floor for any model you build. (`recon/spectrum.py` explains it.)

In [ ]:
vae = load_model('../checkpoints/vae_128.eqx', latent_dim=128)
P = estimate_power_spectrum(train.slices)          # the power-spectrum prior, learnt from the train split

methods = {
    "zero-filled": ev.zero_filled_recon,
    "TV/L1 (FISTA)": ev.tv_recon(lam=1e-3, n_iter=50),
    "power spectrum (Wiener)": ev.wiener_recon(P, n_samples=32),
    "VAE · MAP": ev.map_recon(vae.decoder, vae.latent_dim, steps=600, lr=2e-2),
    "VAE · posterior (NUTS)": ev.posterior_recon(
        vae.decoder, vae.latent_dim, num_samples=NUTS_SAMPLES, num_warmup=NUTS_SAMPLES, max_tree_depth=NUTS_TREE_DEPTH
    ),
}

## 3. Run the sweep

Every method, every slice, every R — one measurement per (slice, R), shared by all methods.

In [ ]:
t0 = time.perf_counter()
res = ev.evaluate(methods, images, ACCELERATIONS, sigma=SIGMA, labels=labels, warmup=WARMUP)
print(f"\nsweep took {time.perf_counter() - t0:.0f}s")

## 4. The table and the curve

Read the table as a scientist: the ± is the spread over slices; the header carries the *effective*
acceleration; `seconds` is what the method costs. If two methods are within one ± of each other, say
"comparable", not "better".

In [ ]:
for metric in ("psnr", "ssim", "nmse", "seconds"):
    print(ev.table(res.rows, metric, fmt="{:.3f}" if metric in ("ssim", "nmse") else "{:.2f}"), "\n")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ev.plot_metric_vs_R(res.rows, "psnr", ax=ax[0]); ax[0].set_title("PSNR vs acceleration (held-out)")
ev.plot_metric_vs_R(res.rows, "ssim", ax=ax[1]); ax[1].set_title("SSIM vs acceleration (held-out)")
plt.tight_layout(); fig.savefig("metric_vs_R.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Is the uncertainty honest? Calibration

Only methods that return a `std` appear here. Pixels are pooled over all evaluated slices and binned by
predicted std; a calibrated method has actual |error| rising with predicted std, close to the diagonal.

Two things to expect and to explain in the talk:

- The **power-spectrum prior's std is flat** — a stationary prior is the same everywhere, so it cannot
  say *where* it is unsure. Its curve is a single point.
- The **VAE posterior is usually overconfident**: its std is below the diagonal. The decoder cannot
  represent the held-out slice exactly, and the model has no term for "my prior can't make this image",
  so it reports tight error bars around the nearest image it *can* make. That is model misspecification,
  and it is the most important caveat on any uncertainty map you show. (You saw it in 1-D in prep 5.)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ev.plot_calibration(res, ax=ax); ax.set_title("calibration (pooled over held-out slices)")
fig.savefig("calibration.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Look at the images — the worst case, and what a clinician would check

Numbers hide things. A method can have the best PSNR and still smear the one structure that matters.
For each method look at its **worst** slice, and go through the visual checklist:

- **Anatomy preserved?** cortical bone edges, the cartilage surfaces, the menisci, the ligaments — are
  they in the right place, and are small ones still there?
- **Artefacts?** residual aliasing ghosts, ringing at edges, blur of fine texture, a "plastic" look.
- **Hallucination?** structure in the reconstruction that is *not* in the ground truth. With a learned
  prior this is the failure mode to fear most; the error map and the uncertainty map should light up there.
- **Does the uncertainty map point at the errors?** Compare the std panel with the error panel.

Save the panels — one of them is the "honest failure" slide in the talk.

In [ ]:
for name in ("VAE · posterior (NUTS)", "power spectrum (Wiener)", "zero-filled"):
    fig = ev.worst_case(res, name)
    fig.savefig(f"worst_{name.split()[0].replace('/', '-')}.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Adding your own model

Two routes, both already used above. Pick the one that fits.

**Route A — your model is a prior with a decoder** (a VAE, a diffusion model with a deterministic
sampler, a power-spectrum decoder, …). Write a pure `decode(z) -> (H, W)` image, choose the latent
shape, and reuse the repo's inference:

```python
methods["my prior · MAP"] = ev.map_recon(decode, latent_shape, steps=600)
methods["my prior · posterior"] = ev.posterior_recon(decode, latent_shape, num_samples=NUTS_SAMPLES, num_warmup=NUTS_SAMPLES)
```

**Route B — your model is its own reconstruction algorithm** (a classical solver, a diffusion
posterior sampler such as DPS, an unrolled network, …). Write the adapter:

```python
def my_recon(y_obs, mask, sigma):
    x_hat = ...                      # your method; use y_obs, mask and sigma, nothing else
    return ev.Recon(mean=np.asarray(x_hat), std=None)    # add std / samples if you have them
methods["my method"] = my_recon
```

Then re-run section 3 **on the same slices, R and σ** and report the same table. Rules that make the
comparison count:

1. Train or fit your model on `split="train"` only, and say what it saw.
2. Report `R_eff`, `seconds`, and the ± over slices, next to everyone else's.
3. If your model gives samples or a std, include it in the calibration plot; if it does not, say so —
   "no uncertainty" is a property of the method, not a missing column.
4. Show its worst case.
5. Evaluate the **prior itself** too (notebook 02): sample quality, diversity, time per sample. A prior
   can win on reconstruction and lose on diversity, or the other way around; both are results.

The cell below demonstrates route A with the power-spectrum prior as a *decoder* — the same prior as
the Wiener row, now reconstructed by the repo's MAP machinery with latent shape `(2, H, W)`. The two
rows should agree (the Wiener filter *is* this model's exact posterior mean), which is the check that
the machinery treats every model the same way.

In [ ]:
H, W = images.shape[1:]
route_a = {
    "power spectrum (Wiener)": ev.wiener_recon(P),
    "power spectrum (MAP via decoder)": ev.map_recon(make_spectrum_decoder(P), (2, H, W), steps=400, lr=5e-2),
}
res_a = ev.evaluate(route_a, images[:1], (4,), sigma=SIGMA, labels=labels[:1], warmup=WARMUP)
print(ev.table(res_a.rows, "psnr"))

## 8. The checklist for the talk

Your evaluation section should be able to answer every line:

- **Quantitative:** the PSNR/SSIM/NMSE table (mean ± std over N held-out slices) at each R_eff, and the
  PSNR-vs-R curve. Which method wins where? By more than the ±?
- **Cost:** seconds per reconstruction. Is the win worth the time? (Zero-filled is free; NUTS is not.)
- **Uncertainty:** the calibration plot; is the std honest or overconfident, and where?
- **Visual / clinical:** one panel where the learned prior clearly helps, and one **honest failure** —
  the worst case, with what went wrong (aliasing left, blur, hallucination).
- **The prior itself:** samples, diversity, time per sample (notebook 02).
- **Honesty:** held-out slices; same mask/noise for all; effective R; seeds fixed; what the checkpoint
  was trained on; what you would do next.

## Done when

`metric_vs_R.png`, `calibration.png` and the worst-case panels exist; you can say which method wins at
each R and whether the difference is larger than the spread; and you can explain why the power-spectrum
prior cannot beat zero-filling on PSNR, and what a well-trained VAE prior has that it lacks. (A VAE trained
on a handful of slices may lose to zero-filling too; that is a result about the prior, and worth saying.)